# Experiment 8: targeted FinBERT encoder fine-tuning

This experiment fine-tunes the complete pretrained `yiyanghkust/finbert-pretrain` encoder. A warm-up model identifies low-confidence hard training examples using top-two probability margins. Those examples receive increased loss weight during targeted fine-tuning. The notebook reports validation/test macro-F1, including hard-example metrics.

## Imports and reproducibility

In [ ]:
import copy
import os
import random
import sys

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, f1_score
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset
from transformers import BertConfig, BertForSequenceClassification, BertTokenizer, get_linear_schedule_with_warmup

sys.path.append("..")
from utils.hard_samples_identify import get_hard_sample_indices
from utils.preprocessing import LABELS, MAX_LEN, encode_labels

HF_CACHE_DIR = os.path.abspath("../.cache/huggingface")
os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["HF_HUB_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")
os.environ["HF_TOKEN_PATH"] = os.path.join(HF_CACHE_DIR, "token")
os.makedirs(os.environ["HF_HUB_CACHE"], exist_ok=True)

SEED = 42
MODEL_NAME = "yiyanghkust/finbert-pretrain"
MAX_LENGTH = MAX_LEN
BATCH_SIZE = 16
WARMUP_EPOCHS = 2
TARGETED_EPOCHS = 4
LEARNING_RATE = 2e-5
HARD_SAMPLE_PERCENTILE = 20
HARD_SAMPLE_WEIGHT = 2.0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FINBERT_CONFIG = BertConfig(
    vocab_size=30873, hidden_size=768, num_hidden_layers=12,
    num_attention_heads=12, intermediate_size=3072, hidden_act="gelu",
    hidden_dropout_prob=0.1, attention_probs_dropout_prob=0.1,
    max_position_embeddings=512, type_vocab_size=2,
    num_labels=len(LABELS),
    id2label={index: label for index, label in enumerate(LABELS)},
    label2id={label: index for index, label in enumerate(LABELS)},
)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

set_seed()
print("Device:", DEVICE)


## Load data and tokenize

In [ ]:
train_df = pd.read_csv("../data/train.csv")
val_df = pd.read_csv("../data/val.csv")
test_df = pd.read_csv("../data/test.csv")
y_train = encode_labels(train_df.label)
y_val = encode_labels(val_df.label)
y_test = encode_labels(test_df.label)

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME, cache_dir=HF_CACHE_DIR, token=False, use_fast=False,)

def tokenize(frame):
    encoded = tokenizer(
        frame.sentence.tolist(), padding="max_length", truncation=True,
        max_length=MAX_LENGTH, return_tensors="pt",
    )
    return encoded["input_ids"], encoded["attention_mask"]

input_train, mask_train = tokenize(train_df)
input_val, mask_val = tokenize(val_df)
input_test, mask_test = tokenize(test_df)
print(f"model={MODEL_NAME} vocab={FINBERT_CONFIG.vocab_size}")
print(input_train.shape, input_val.shape, input_test.shape)


## Training and evaluation helpers

In [ ]:
def loader(input_ids, attention_mask, labels, weights=None, shuffle=False):
    tensors = [input_ids, attention_mask, torch.tensor(labels, dtype=torch.long)]
    if weights is not None:
        tensors.append(torch.tensor(weights, dtype=torch.float32))
    return DataLoader(TensorDataset(*tensors), batch_size=BATCH_SIZE, shuffle=shuffle)


def new_model():
    set_seed()
    model = BertForSequenceClassification.from_pretrained(MODEL_NAME, config=FINBERT_CONFIG, cache_dir=HF_CACHE_DIR, token=False, ignore_mismatched_sizes=True,)
    return model.to(DEVICE)


@torch.no_grad()
def predict(model, input_ids, attention_mask, labels):
    model.eval(); probabilities=[]; predictions=[]; losses=[]
    for batch in loader(input_ids, attention_mask, labels):
        ids, masks, batch_labels = [value.to(DEVICE) for value in batch]
        output = model(input_ids=ids, attention_mask=masks, labels=batch_labels)
        probabilities.append(torch.softmax(output.logits, dim=1).cpu().numpy())
        predictions.append(output.logits.argmax(dim=1).cpu().numpy())
        losses.append(output.loss.item())
    return np.concatenate(probabilities), np.concatenate(predictions), float(np.mean(losses))


def fit_model(model, input_ids, attention_mask, labels, validation_data, epochs, sample_weights=None):
    train_loader = loader(input_ids, attention_mask, labels, sample_weights, shuffle=True)
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, max(1, steps // 10), steps)
    best_state = copy.deepcopy(model.state_dict()); best_score = -np.inf
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            ids, masks, batch_labels, *extra = [value.to(DEVICE) for value in batch]
            output = model(input_ids=ids, attention_mask=masks, labels=batch_labels)
            if extra:
                per_example = torch.nn.functional.cross_entropy(output.logits, batch_labels, reduction="none")
                loss = (per_example * extra[0]).mean()
            else:
                loss = output.loss
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
        _, val_predictions, val_loss = predict(model, *validation_data)
        val_score = f1_score(validation_data[2], val_predictions, average="macro")
        print(f"epoch {epoch + 1}/{epochs} - val_loss: {val_loss:.4f} - val_macro_f1: {val_score:.4f}")
        if val_score > best_score:
            best_score = val_score; best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model


def score(model, input_ids, attention_mask, labels, hard_indices):
    probabilities, predictions, _ = predict(model, input_ids, attention_mask, labels)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "hard_accuracy": accuracy_score(labels[hard_indices], predictions[hard_indices]),
        "hard_macro_f1": f1_score(labels[hard_indices], predictions[hard_indices], average="macro"),
    }, probabilities, predictions


## Train the targeted transformer


In [ ]:
validation_data = (input_val, mask_val, y_val)
warmup_model = fit_model(new_model(), input_train, mask_train, y_train, validation_data, epochs=WARMUP_EPOCHS,)
train_warmup_probs, _, _ = predict(warmup_model, input_train, mask_train, y_train)
val_warmup_probs, _, _ = predict(warmup_model, input_val, mask_val, y_val)
test_warmup_probs, _, _ = predict(warmup_model, input_test, mask_test, y_test)

train_hard_indices, train_cutoff = get_hard_sample_indices(train_warmup_probs, HARD_SAMPLE_PERCENTILE)
val_hard_indices, val_cutoff = get_hard_sample_indices(val_warmup_probs, HARD_SAMPLE_PERCENTILE)
test_hard_indices, test_cutoff = get_hard_sample_indices(test_warmup_probs, HARD_SAMPLE_PERCENTILE)
print(f"Hard examples P{HARD_SAMPLE_PERCENTILE}: train={len(train_hard_indices)} val={len(val_hard_indices)} test={len(test_hard_indices)}")

sample_weights = np.ones(len(y_train), dtype=np.float32)
sample_weights[train_hard_indices] = HARD_SAMPLE_WEIGHT
targeted_transformer = fit_model(
    warmup_model, input_train, mask_train, y_train, validation_data,
    epochs=TARGETED_EPOCHS, sample_weights=sample_weights,
)


## Evaluate the targeted transformer


## Model size


In [27]:
total_parameters = sum(parameter.numel() for parameter in targeted_transformer.parameters())
trainable_parameters = sum(
    parameter.numel() for parameter in targeted_transformer.parameters()
    if parameter.requires_grad
)

print(f"Total parameters: {total_parameters:,} ({total_parameters / 1e6:.2f}M)")
print(f"Trainable parameters: {trainable_parameters:,} ({trainable_parameters / 1e6:.2f}M)")
print(f"Estimated FP32 weights: {total_parameters * 4 / 1024**2:.1f} MiB")
print(f"Estimated FP16 weights: {total_parameters * 2 / 1024**2:.1f} MiB")
print(f"Estimated INT8 weights: {total_parameters / 1024**2:.1f} MiB")


Total parameters: 109,754,115 (109.75M)
Trainable parameters: 109,754,115 (109.75M)
Estimated FP32 weights: 418.7 MiB
Estimated FP16 weights: 209.3 MiB
Estimated INT8 weights: 104.7 MiB


## Report targeted transformer statistics


In [28]:
validation_metrics, _, validation_predictions = score(targeted_transformer, input_val, mask_val, y_val, val_hard_indices)
test_metrics, _, test_predictions = score(targeted_transformer, input_test, mask_test, y_test, test_hard_indices)
results_df = pd.DataFrame([
    {"split": "validation", "hard_count": len(val_hard_indices), "hard_cutoff": val_cutoff, **validation_metrics},
    {"split": "test", "hard_count": len(test_hard_indices), "hard_cutoff": test_cutoff, **test_metrics},
])
print("Targeted FinBERT transformer statistics")
print(results_df.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
print("\nValidation classification report")
print(classification_report(y_val, validation_predictions, target_names=LABELS))
print("Test classification report")
print(classification_report(y_test, test_predictions, target_names=LABELS))


Targeted FinBERT transformer statistics
     split  hard_count  hard_cutoff  accuracy  macro_f1  hard_accuracy  hard_macro_f1
validation         194       0.6681    0.8428    0.8220         0.6031         0.5826
      test         194       0.6448    0.8574    0.8390         0.5825         0.5601

Validation classification report
              precision    recall  f1-score   support

    negative       0.83      0.78      0.80       121
     neutral       0.88      0.88      0.88       574
    positive       0.78      0.79      0.78       272

    accuracy                           0.84       967
   macro avg       0.83      0.82      0.82       967
weighted avg       0.84      0.84      0.84       967

Test classification report
              precision    recall  f1-score   support

    negative       0.79      0.85      0.82       121
     neutral       0.89      0.89      0.89       575
    positive       0.82      0.79      0.81       272

    accuracy                           0.8

In [29]:
## Report warm-up transformer statistics

warmup_validation_metrics, _, warmup_validation_predictions = score(
    warmup_model,
    input_val,
    mask_val,
    y_val,
    val_hard_indices,
)

warmup_test_metrics, _, warmup_test_predictions = score(
    warmup_model,
    input_test,
    mask_test,
    y_test,
    test_hard_indices,
)

warmup_results_df = pd.DataFrame([
    {
        "split": "validation",
        "hard_count": len(val_hard_indices),
        "hard_cutoff": val_cutoff,
        **warmup_validation_metrics,
    },
    {
        "split": "test",
        "hard_count": len(test_hard_indices),
        "hard_cutoff": test_cutoff,
        **warmup_test_metrics,
    },
])

print("Warm-up FinBERT transformer statistics")
print(
    warmup_results_df.to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}",
    )
)

print("\nWarm-up validation classification report")
print(
    classification_report(
        y_val,
        warmup_validation_predictions,
        target_names=LABELS,
    )
)

print("Warm-up test classification report")
print(
    classification_report(
        y_test,
        warmup_test_predictions,
        target_names=LABELS,
    )
)

Warm-up FinBERT transformer statistics
     split  hard_count  hard_cutoff  accuracy  macro_f1  hard_accuracy  hard_macro_f1
validation         194       0.6681    0.8428    0.8220         0.6031         0.5826
      test         194       0.6448    0.8574    0.8390         0.5825         0.5601

Warm-up validation classification report
              precision    recall  f1-score   support

    negative       0.83      0.78      0.80       121
     neutral       0.88      0.88      0.88       574
    positive       0.78      0.79      0.78       272

    accuracy                           0.84       967
   macro avg       0.83      0.82      0.82       967
weighted avg       0.84      0.84      0.84       967

Warm-up test classification report
              precision    recall  f1-score   support

    negative       0.79      0.85      0.82       121
     neutral       0.89      0.89      0.89       575
    positive       0.82      0.79      0.81       272

    accuracy               

## Save the targeted transformer


In [30]:
metrics = pd.read_csv("../results/metrics.csv")

transformer_row = {
    "condition": "targeted",
    "embedding_config": "finbert_pretrained",
    "architecture": "transformer_finbert",
    "accuracy_overall": test_metrics["accuracy"],
    "f1_overall": test_metrics["macro_f1"],
    "accuracy_hard": test_metrics["hard_accuracy"],
    "f1_hard": test_metrics["hard_macro_f1"],
}

metrics = pd.concat([metrics, pd.DataFrame([transformer_row])], ignore_index=True)
metrics.to_csv("../results/metrics.csv", index=False)

torch.save(
    {
        "state_dict": targeted_transformer.state_dict(),
        "config": FINBERT_CONFIG.to_dict(),
    },
    "../models/experiment_8_finbert_targeted_transformer.pt",
)